# Bund/ESTR invoice spread (RXAISPE) — rich/cheap monitor

Fair-value model for the **RX (Bund) future invoice spread vs ESTR** (`RXAISPE`).

**Method**

1. Fetch the target spread and a block of explanatory factors (funding/forward
   rates, Euribor–ESTR basis, curve slope, Bund yield, MOVE, UST invoice
   spread, BTP–Bund, 3m10y swaption vol).
2. Diagnostics: covariance / correlation / beta of each factor's daily changes
   vs the Bund spread's daily changes (full sample and trailing window).
3. Fair value via **rolling PCR**: each day, PCA the standardised factor block
   over the trailing window, regress the spread level on the top-K principal
   components, and project today's factor observation → model fair value.
4. Residual = market − fair, smoothed with a Kalman filter, then **z-scored**
   on a rolling window so you can see the full rich/cheap history.

**Sign convention:** `z > 0` → spread trades **above** model fair (wide/CHEAP
to buy), `z < 0` → **below** fair (tight/RICH). Flip the labels mentally if
you quote the spread the other way round.

**How to run:** open in BQuant on the Terminal and *Run All Cells*. The last
cell renders the live app (Refresh button re-fetches and recomputes).
This notebook cannot be executed outside BQuant, so it is untested here —
if the data request fails, check the ticker yellow-keys in the CONFIG cell
first (e.g. `RXAISPE Comdty` vs `RXAISPE Index`, and your preferred
swaption-vol ticker).


In [ ]:
# =============================== CONFIG =====================================
TARGET_TICKER = "RXAISPE Comdty"      # Bund future invoice spread vs ESTR (bp)
                                      # if the request errors, try "RXAISPE Index"

# Raw tickers fetched. ER2/TKY2 are combined into one basis factor below.
RAW_TICKERS = {
    "EEFOSC2":    "EEFOSC2 Curncy",   # ECB-dated ESTR OIS forward (2nd meeting)
    "ER2":        "ER2 Comdty",       # 3M Euribor future, 2nd contract
    "TKY2":       "TKY2 Comdty",      # 3M ESTR future, 2nd contract
    "EUSS0210":   "EUSS0210 Curncy",  # EUR 2s10s swap slope
    "GTDEM10Y":   "GTDEM10Y Govt",    # German 10Y generic yield
    "MOVE":       "MOVE Index",       # US rates implied vol
    "UXYAISPE":   "UXYAISPE Comdty",  # UST Ultra-10 future invoice spread
    "DBRBTP10Y":  "DBRBTP10Y Index",  # 10Y BTP-Bund spread (has a fallback below)
    "SWPT_3M10Y": "EUSV0310 Curncy",  # 3m10y EUR swaption vol — swap for your
                                      # preferred vol ticker (e.g. normal vol)
}
# used to derive BTP-Bund if DBRBTP10Y is not returned
FALLBACK_BTP, FALLBACK_BUND = "GBTPGR10 Index", "GDBR10 Index"

HISTORY      = "-8Y"     # history requested from BQL
ROLL_WIN     = 252       # rolling PCA/regression window (business days)
N_PCS        = 4         # principal components kept in the fair-value regression
Z_WIN        = 252       # rolling window for the z-score std
MIN_COVERAGE = 0.60      # drop factors with less non-NaN coverage
FFILL_LIMIT  = 5         # max days a stale factor value is carried forward
KALMAN_PHI   = 0.97      # AR(1) persistence of the residual state
KALMAN_Q     = 0.02      # state noise
TRADE_Z, WATCH_Z = 2.0, 1.5
PLOT_OBS     = 756       # ~3y shown on the market-vs-fair chart


## Data layer

In [ ]:
import numpy as np
import pandas as pd
import bql

bq = bql.Service()


def bql_levels(tickers, hist=HISTORY):
    """Wide DataFrame of px_last: index=date, one column per ticker (upper)."""
    px = bq.data.px_last(dates=bq.func.range(hist, "0d"), fill="prev")
    df = bq.execute(bql.Request(list(tickers), {"px": px}))[0].df().reset_index()
    df.columns = [str(c).upper() for c in df.columns]
    idc = next(c for c in df.columns if c in ("ID", "SECURITY", "SECURITIES"))
    dc = next(c for c in df.columns if "DATE" in c)
    vc = "PX" if "PX" in df.columns else next(
        c for c in df.columns if c not in (idc, dc, "CURRENCY"))
    df[dc] = pd.to_datetime(df[dc])
    df[vc] = pd.to_numeric(df[vc], errors="coerce")
    wide = df.pivot_table(index=dc, columns=idc, values=vc, aggfunc="last")
    wide.columns = [str(c).upper() for c in wide.columns]
    wide = wide.sort_index()
    return wide[wide.index.weekday < 5]


def fetch_all():
    """One request for target + factors + fallbacks. Returns (wide, missing)."""
    wanted = [TARGET_TICKER] + list(RAW_TICKERS.values()) + [FALLBACK_BTP, FALLBACK_BUND]
    wide = bql_levels(sorted(set(wanted)))
    missing = [t for t in wanted if t.upper() not in wide.columns
               or wide[t.upper()].notna().sum() == 0]
    return wide, missing


## Factor construction

`ER2 − TKY2` is turned into the **Euribor–ESTR basis** of the 2nd contract:
both futures quote `100 − rate`, so `(TKY2 − ER2) × 100` is the basis in bp.
`DBRBTP10Y` falls back to `(GBTPGR10 − GDBR10) × 100` if the spread ticker
is not available.


In [ ]:
def build_dataset(wide):
    """Return (target Series in bp, factor DataFrame) aligned on dates."""
    def col(tk):
        return wide[tk.upper()] if tk.upper() in wide.columns else pd.Series(
            np.nan, index=wide.index)

    target = col(TARGET_TICKER).rename("RXAISPE")

    fac = pd.DataFrame(index=wide.index)
    fac["EEFOSC2"] = col(RAW_TICKERS["EEFOSC2"])
    # Euribor-ESTR basis, bp (both futures quote 100 - rate)
    fac["ER2_TKY2"] = (col(RAW_TICKERS["TKY2"]) - col(RAW_TICKERS["ER2"])) * 100.0
    fac["EUSS0210"] = col(RAW_TICKERS["EUSS0210"])
    fac["GTDEM10Y"] = col(RAW_TICKERS["GTDEM10Y"])
    fac["MOVE"] = col(RAW_TICKERS["MOVE"])
    fac["UXYAISPE"] = col(RAW_TICKERS["UXYAISPE"])

    btp = col(RAW_TICKERS["DBRBTP10Y"])
    if btp.notna().mean() < MIN_COVERAGE:
        btp = (col(FALLBACK_BTP) - col(FALLBACK_BUND)) * 100.0
    fac["BTP_BUND"] = btp

    fac["SWPT_3M10Y"] = col(RAW_TICKERS["SWPT_3M10Y"])

    # coverage guard, then align to the target's dates
    keep = [c for c in fac.columns if fac[c].notna().mean() >= MIN_COVERAGE]
    dropped = [c for c in fac.columns if c not in keep]
    fac = fac[keep].reindex(target.index).ffill(limit=FFILL_LIMIT)

    good = target.notna() & fac.notna().all(axis=1)
    return target[good], fac[good], dropped


## Covariance diagnostics vs the Bund spread

Daily-change covariance, correlation and univariate beta of each factor
against the target, full sample and trailing `ROLL_WIN`.


In [ ]:
def covariance_table(target, fac, win=ROLL_WIN):
    dt, df_ = target.diff(), fac.diff()
    rows = []
    for c in fac.columns:
        full = pd.concat([dt, df_[c]], axis=1).dropna()
        tail = full.tail(win)
        def stats(d):
            cov = d.iloc[:, 0].cov(d.iloc[:, 1])
            var = d.iloc[:, 1].var()
            return cov, d.iloc[:, 0].corr(d.iloc[:, 1]), (cov / var if var else np.nan)
        cov_f, cor_f, b_f = stats(full)
        cov_t, cor_t, b_t = stats(tail)
        rows.append({"Factor": c,
                     "Cov (full)": cov_f, "Corr (full)": cor_f, "Beta (full)": b_f,
                     f"Cov ({win}d)": cov_t, f"Corr ({win}d)": cor_t,
                     f"Beta ({win}d)": b_t})
    return pd.DataFrame(rows).set_index("Factor").round(4)


## Fair-value model: rolling PCR + Kalman-smoothed residual z-score

Each day the standardised factor block over the trailing window is decomposed
with PCA; the spread level is regressed on the top-`N_PCS` components and
today's observation is projected through the fit → fair value. The per-factor
betas (bp per 1σ factor move) are recovered from the PC loadings for
attribution.


In [ ]:
def fair_value_pcr(fac, target, win=ROLL_WIN, k=N_PCS):
    """Rolling PCR fair value. Returns (fair Series, per-factor beta DataFrame,
    beta units = bp of spread per 1 trailing-window sigma of the factor)."""
    common = fac.dropna().index.intersection(target.dropna().index)
    X, y = fac.loc[common].values, target.loc[common].values
    n, m = X.shape
    k = min(k, m)
    fair = np.full(n, np.nan)
    betas = np.full((n, m), np.nan)
    for i in range(win, n):
        Xw, yw = X[i - win:i], y[i - win:i]
        mu, sd = Xw.mean(0), Xw.std(0)
        sd[sd == 0] = 1.0
        Xz, xi = (Xw - mu) / sd, (X[i] - mu) / sd
        e, V = np.linalg.eigh(np.cov(Xz, rowvar=False))
        Vk = V[:, np.argsort(e)[::-1][:k]]
        A = np.column_stack([np.ones(win), Xz @ Vk])
        b, *_ = np.linalg.lstsq(A, yw, rcond=None)
        fair[i] = b[0] + (xi @ Vk) @ b[1:]
        betas[i] = Vk @ b[1:]
    fair = pd.Series(fair, index=common, name="fair").reindex(target.index)
    betas = pd.DataFrame(betas, index=common, columns=fac.columns).reindex(target.index)
    return fair, betas


def kalman(s, phi=KALMAN_PHI, Q=KALMAN_Q, P0=1.0):
    """AR(1) Kalman smoother; observation noise from the residual's own diffs."""
    y = s.values.astype(float)
    v = y[~np.isnan(y)]
    R = max(0.5 * np.var(np.diff(v)), 1e-4) if len(v) > 5 else 1.0
    x = np.zeros(len(y))
    xi, pi = 0.0, P0
    for t in range(len(y)):
        xp, pp = phi * xi, phi * pi * phi + Q
        if np.isnan(y[t]):
            xi, pi = xp, pp
        else:
            Kg = pp / (pp + R)
            xi = xp + Kg * (y[t] - xp)
            pi = (1 - Kg) * pp
        x[t] = xi
    return pd.Series(x, index=s.index)


def compute():
    """Full pipeline: fetch -> factors -> fair value -> z. Returns a dict."""
    wide, missing = fetch_all()
    target, fac, dropped = build_dataset(wide)
    if len(target) < ROLL_WIN + Z_WIN:
        raise RuntimeError(
            f"only {len(target)} aligned observations - need "
            f"{ROLL_WIN + Z_WIN}; check tickers/history")

    fair, betas = fair_value_pcr(fac, target)
    resid = (target - fair).dropna()
    resid_s = kalman(resid)
    z = (resid_s / resid_s.rolling(Z_WIN).std()).dropna()

    zl = float(z.iloc[-1])
    signal = ("CHEAP (wide vs model)" if zl >= TRADE_Z else
              "RICH (tight vs model)" if zl <= -TRADE_Z else
              "watch - cheap side" if zl >= WATCH_Z else
              "watch - rich side" if zl <= -WATCH_Z else "NEUTRAL")

    return {"target": target, "fair": fair, "resid": resid, "resid_s": resid_s,
            "z": z, "z_last": zl, "signal": signal, "betas": betas,
            "cov": covariance_table(target, fac), "fac": fac,
            "missing": missing, "dropped": dropped,
            "asof": target.index[-1]}


## One-off run (tables)

Runs the pipeline once and prints the covariance table and the latest
per-factor betas — useful before launching the app.


In [ ]:
RES = compute()

print(f"as of {RES['asof']:%Y-%m-%d}   "
      f"market {RES['target'].iloc[-1]:+.1f}  fair {RES['fair'].iloc[-1]:+.1f}  "
      f"gap {RES['target'].iloc[-1] - RES['fair'].iloc[-1]:+.1f} bp   "
      f"z {RES['z_last']:+.2f}  ->  {RES['signal']}")
if RES["missing"]:
    print("WARNING - tickers returned no data:", RES["missing"])
if RES["dropped"]:
    print("factors dropped for low coverage:", RES["dropped"])

print("\nCovariance / correlation / beta of daily changes vs RXAISPE:")
display(RES["cov"])

print(f"\nLatest per-factor betas (bp of spread per 1 sigma of factor, "
      f"{ROLL_WIN}d window):")
display(RES["betas"].dropna().iloc[-1].round(2).to_frame("beta_bp_per_sigma"))


## Live app

Number card (market / fair / gap / z), market-vs-fair chart, full z-score
history with trade/watch bands, and the current per-factor beta attribution.
**Refresh** re-fetches and recomputes in place.


In [ ]:
import ipywidgets as widgets
import plotly.graph_objects as go
from datetime import datetime
from IPython.display import display

try:
    from zoneinfo import ZoneInfo
    APP_TZ = ZoneInfo("Europe/London")
except Exception:
    APP_TZ = None

DARK, GRID = "#141412", "#2a2a26"

# --- widgets built once -----------------------------------------------------
btn = widgets.Button(description="Refresh", button_style="primary", icon="refresh")
spinner = widgets.HTML('<i class="fa fa-spinner fa-spin"></i>',
                       layout={"visibility": "hidden"})
status = widgets.HTML("")
card = widgets.HTML("")

fig_fv = go.FigureWidget()
fig_fv.add_scatter(x=[], y=[], name="market", line=dict(color="#eda100", width=1.8))
fig_fv.add_scatter(x=[], y=[], name="model fair", line=dict(color="#5aa9e6", width=1.4))
fig_fv.update_layout(template="plotly_dark", height=300, width=560,
                     title=f"{TARGET_TICKER} - market vs model fair (bp)",
                     margin=dict(t=36, l=48, r=14, b=28), font=dict(size=10),
                     paper_bgcolor=DARK, plot_bgcolor=DARK,
                     legend=dict(orientation="h", y=1.12))

fig_z = go.FigureWidget()
fig_z.add_scatter(x=[], y=[], name="z", line=dict(color="#eda100", width=1.5))
for lv, col, dash in [(TRADE_Z, "#30a46c", "dash"), (WATCH_Z, "#8a8a84", "dot"),
                      (0, "#8a8a84", None),
                      (-WATCH_Z, "#8a8a84", "dot"), (-TRADE_Z, "#e5484d", "dash")]:
    fig_z.add_hline(y=lv, line=dict(color=col, width=1, dash=dash))
fig_z.add_annotation(xref="paper", x=0.01, y=TRADE_Z, text="CHEAP (wide)",
                     showarrow=False, font=dict(size=9, color="#30a46c"), yshift=8)
fig_z.add_annotation(xref="paper", x=0.01, y=-TRADE_Z, text="RICH (tight)",
                     showarrow=False, font=dict(size=9, color="#e5484d"), yshift=-8)
fig_z.update_layout(template="plotly_dark", height=300, width=560,
                    title="residual z-score history", showlegend=False,
                    margin=dict(t=36, l=48, r=14, b=28), font=dict(size=10),
                    paper_bgcolor=DARK, plot_bgcolor=DARK)

fig_beta = go.FigureWidget(data=[go.Bar(
    x=[], y=[], marker_color=[],
    hovertemplate="%{x}: %{y:.2f} bp/sigma<extra></extra>")])
fig_beta.update_layout(template="plotly_dark", height=280, width=1130,
                       title=f"per-factor beta (bp of spread per 1 sigma of factor, "
                             f"{ROLL_WIN}d PCR window)",
                       margin=dict(t=36, l=48, r=14, b=60), font=dict(size=10),
                       paper_bgcolor=DARK, plot_bgcolor=DARK, showlegend=False)
fig_beta.update_xaxes(tickangle=30)


def _card_html(r):
    mkt = r["target"].iloc[-1]
    fair = r["fair"].iloc[-1]
    gap, z = mkt - fair, r["z_last"]
    zc = ("#30a46c" if z >= TRADE_Z else "#e5484d" if z <= -TRADE_Z else "#c9c8c2")
    row = lambda l, v, c="#c9c8c2": (
        f'<div style="display:flex;justify-content:space-between;margin:2px 0">'
        f'<span style="color:#8a8a84">{l}</span>'
        f'<span style="color:{c};font-variant-numeric:tabular-nums">{v}</span></div>')
    return (
        '<div style="background:linear-gradient(160deg,#1c1c19,#141412);'
        'border:1px solid #2a2a26;border-radius:10px;padding:14px 16px;'
        'width:250px;font-family:sans-serif;font-size:12px">'
        f'<div style="color:#eda100;font-weight:600;margin-bottom:6px">'
        f'{TARGET_TICKER}</div>'
        + row("market", f"{mkt:+.1f} bp")
        + row("model fair", f"{fair:+.1f} bp")
        + row("gap", f"{gap:+.1f} bp")
        + row("z-score", f"{z:+.2f}", zc)
        + f'<div style="margin-top:8px;color:{zc};font-weight:600">{r["signal"]}</div>'
        f'<div style="margin-top:6px;color:#8a8a84;font-size:10px">'
        f'as of {r["asof"]:%d %b %Y}</div></div>')


def run(_=None):
    spinner.layout.visibility = "visible"
    status.value = '<span style="color:#888">loading...</span>'
    try:
        r = compute()
        card.value = _card_html(r)
        tail = min(PLOT_OBS, len(r["target"]))
        with fig_fv.batch_update():
            fig_fv.data[0].x = list(r["target"].index[-tail:])
            fig_fv.data[0].y = list(r["target"].values[-tail:])
            fig_fv.data[1].x = list(r["fair"].index[-tail:])
            fig_fv.data[1].y = list(r["fair"].values[-tail:])
        with fig_z.batch_update():
            fig_z.data[0].x = list(r["z"].index)
            fig_z.data[0].y = list(r["z"].values)
        b = r["betas"].dropna().iloc[-1].sort_values()
        with fig_beta.batch_update():
            fig_beta.data[0].x = list(b.index)
            fig_beta.data[0].y = list(b.values)
            fig_beta.data[0].marker.color = [
                "#30a46c" if v > 0 else "#e5484d" for v in b.values]
        note = (f' - <span style="color:#d6a100">no data: '
                f'{", ".join(r["missing"])}</span>' if r["missing"] else "")
        stamp = f"{datetime.now(APP_TZ):%H:%M:%S}" if APP_TZ else \
                f"{datetime.now():%H:%M:%S}"
        status.value = (f'<span style="color:#888">ok - {len(r["z"])} obs - '
                        f'refreshed {stamp}{note}</span>')
    except Exception as e:
        status.value = f'<span style="color:#d62728">Error: {e}</span>'
    finally:
        spinner.layout.visibility = "hidden"


btn.on_click(run)
app = widgets.VBox([
    widgets.HBox([btn, spinner, status]),
    widgets.HBox([card, fig_fv, fig_z]),
    fig_beta,
])
display(app)
run()
